# trust — SC Linux

Import a verified public IPA CA. Never import a private key.

Use a working copy created by `make lab` for interactive execution. For interactive execution, edit `NOTEBOOK_ARGS` in the next cell. CI passes these arguments through `scripts/notebook.sh`.

## Locate the repository

Jupyter kernels may start in `notebooks/` or `build/notebooks/`. Use the repository root for all relative paths.

In [ ]:
from pathlib import Path as _SCPath
import os as _sc_os
_start = _SCPath.cwd().resolve()
_candidates = (_start, *_start.parents)
_project_root = next(
    (path for path in _candidates if (path / "config/build.json").is_file() and (path / "notebooks").is_dir()),
    None,
)
if _project_root is None:
    raise RuntimeError("Open this notebook from inside the SC Linux checkout")
_sc_os.chdir(_project_root)
print("Repository:", _project_root)

In [ ]:
NOTEBOOK_ARGS = globals().get('NOTEBOOK_ARGS', [])

## Setup

In [ ]:
"""Install a public IPA CA only after checking an independently obtained fingerprint."""

import argparse
import pathlib
import sys

ROOT = pathlib.Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "lib"))
from scos.core import verify_ca

### main

In [ ]:
def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("certificate", type=pathlib.Path)
    p.add_argument(
        "--sha256",
        required=True,
        help="Independently verified SHA-256 of the DER certificate",
    )
    args = p.parse_args()
    digest = verify_ca(args.certificate, args.sha256)
    raw = args.certificate.read_bytes()
    (ROOT / "trust/ipa-ca.crt").write_bytes(raw)
    (ROOT / "trust/ipa-ca.sha256").write_text(digest + "\n")
    print(
        "Verified public CA installed. Review and commit trust/ipa-ca.crt and trust/ipa-ca.sha256."
    )

## Execute

The final cell invokes the operation. Any exception means this step failed.

In [ ]:
if __name__ == "__main__":
    import sys
    sys.argv = ['notebooks/trust.ipynb', *NOTEBOOK_ARGS]
    try:
        main()
    except (ValueError, OSError) as e:
        sys.exit(str(e))